In [ ]:
from typing import Any, Self
from pydantic import (
    BaseModel, EmailStr, Field, field_serializer,
    field_validator, model_serializer, model_validator, SecretStr,
)

Agora lidaremos com serialização e desserialização de dados. Em resumo, os dados percorrem a Web em formatos padronizados, como JSON; desserialização é o processo para receber esses dados advindos da rede, e serialização é o processo de formatar esses JSON e envia-los para a rede.

In [ ]:
class Role(enum.IntFlag):
    User = 0
    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8
    
class User(BaseModel):
    # ... (name e email permanecem iguais)
    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user", exclude=True
    )
    role: Role = Field(
        description="The role of the user",
        examples=[1, 2, 4, 8],
        default=0,
        validate_default=True,
    )

Na classe Role, a única mudança foi a adição da role "User", com valor 0, isto é, com nenhuma permissão.
Na classe User, temos duas adições na função `Field()`:
    - O uso de `excluded=True`, ou seja, a senha nunca vai ser incluída na importação de um JSON.
    - `validade_default=True`, ou seja, a role 0 é a padrão e também passará por todas as validações.

In [ ]:
@model_validator(mode="before")
    @classmethod
    def validate_user_pre(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v:
            raise ValueError("Name and password are required")
        if v["name"].casefold() in v["password"].casefold():
            raise ValueError("Password cannot contain name")
        if not VALID_PASSWORD_REGEX.match(v["password"]):
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest()
        return v

Código idêntico ao do exemplo anterior.

In [ ]:
@model_validator(mode="after")
    def validate_user_post(self, v: Any) -> Self:
        if self.role == Role.Admin and self.name != "Arjan":
            raise ValueError("Only Arjan can be an admin")
        return self

Em resumo, ele olha o "v" (objeto recém recebido) e entende que, se a Role for Admin e o Nome não for "Arjan", ele retorna um `ValueError`, dado que só Arjan pode ser admin nesse sistema.

Nota-se o uso do `mode="after"`, o que significa que, na chamada dessa função, o objeto `User` já foi criado.

In [ ]:
@field_serializer("role", when_used="json")
    @classmethod
    def serialize_role(cls, v) -> str:
        return v.name

Se enviarmos o `IntFlag` do cargo para um site em JSON, ele vai mandar um número (ex: 4). O frontend pode não saber o que 4 significa. O `@field_serializer` intercepta a exportação desse campo e traduz para uma string (ex: "Admin"), facilitando a vida de quem vai consumir essa API. O parâmetro when_used="json" garante que essa tradução só ocorra se estivermos exportando para JSON (se usarmos internamente no Python, ele continua sendo o objeto inteligente Role).

In [ ]:
@model_serializer(mode="wrap", when_used="json")
    def serialize_user(self, serializer, info) -> dict[str, Any]:
        if not info.include and not info.exclude:
            return {"name": self.name, "role": self.role.name}
        return serializer(self)

O `mode="wrap"` significa que estamos "envelopando" a função original de serialização do Pydantic.
A lógica é: se o desenvolvedor pedir para exportar o objeto de forma crua (sem especificar explicitamente o que quer incluir ou excluir), o Pydantic vai retornar um mini-dicionário contendo apenas o nome e o cargo. O e-mail (e obviamente a senha) ficam ocultos por padrão. É uma proteção agressiva para APIs públicas.